In [2]:
# ==========================
# Import Libraries
# ==========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ==========================
# Load Dataset
# ==========================
df = pd.read_csv("ai-impact-jobs-layoff-risk-dataset.csv")

# ==========================
# Check Missing Values
# ==========================
print(df.isnull().sum())

# ==========================
# Encode Categorical Columns
# ==========================
label_encoders = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# ==========================
# Split Features and Target
# ==========================
X = df.drop("Layoff_Risk", axis=1)
y = df["Layoff_Risk"]

# ==========================
# Train Test Split
# ==========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================
# Random Forest Model
# ==========================
rf = RandomForestClassifier(random_state=42)

# ==========================
# Hyperparameter Grid
# ==========================
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

# ==========================
# Randomized Search
# ==========================
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    verbose=2,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(random_search.best_score_)

# ==========================
# Best Model
# ==========================
best_rf = random_search.best_estimator_

# ==========================
# Prediction
# ==========================
y_pred = best_rf.predict(X_test)

# ==========================
# Evaluation
# ==========================
print("\n==============================")
print("Random Forest Performance")
print("==============================")

print("Accuracy :", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

# ==========================
# Feature Importance
# ==========================
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 15 Important Features\n")
print(importance.head(15))

Age                           0
Education_Level               0
Years_of_Experience           0
Industry                      0
Job_Role                      0
Company_Size                  0
Job_Level                     0
Routine_Task_Percentage       0
Creativity_Requirement        0
Human_Interaction_Level       0
AI_Adoption_Level             0
Number_of_AI_Tools_Used       0
AI_Usage_Hours_Per_Week       0
Tasks_Automated_Percentage    0
AI_Training_Hours             0
Layoff_Risk                   0
dtype: int64
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30, 'bootstrap': False}

Best Cross Validation Accuracy:
0.8919375

Random Forest Performance
Accuracy : 0.902

Classification Report

              precision    recall  f1-score   support

           0       0.93      0.93      0.93      1360
           1       0.93      0.93      0.93  

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000, random_state=42)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("="*30)
print("Logistic Regression")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Logistic Regression
Accuracy : 0.8285

              precision    recall  f1-score   support

           0       0.88      0.88      0.88      1360
           1       0.86      0.87      0.86      1320
           2       0.74      0.73      0.74      1320

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000


[[1202    0  158]
 [   0 1142  178]
 [ 166  184  970]]


C:\Users\sahit\.conda\envs\ml\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)

print("="*30)
print("Decision Tree")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Decision Tree
Accuracy : 0.82925

              precision    recall  f1-score   support

           0       0.88      0.85      0.87      1360
           1       0.87      0.88      0.87      1320
           2       0.74      0.76      0.75      1320

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000


[[1162    1  197]
 [   0 1158  162]
 [ 153  170  997]]


In [5]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("="*30)
print("KNN")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

KNN
Accuracy : 0.67225

              precision    recall  f1-score   support

           0       0.72      0.81      0.77      1360
           1       0.74      0.75      0.74      1320
           2       0.53      0.45      0.49      1320

    accuracy                           0.67      4000
   macro avg       0.66      0.67      0.66      4000
weighted avg       0.66      0.67      0.67      4000


[[1105   12  243]
 [  40  985  295]
 [ 383  338  599]]


In [6]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', random_state=42)

svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

print("="*30)
print("Support Vector Machine")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Support Vector Machine
Accuracy : 0.7555

              precision    recall  f1-score   support

           0       0.80      0.87      0.83      1360
           1       0.82      0.79      0.80      1320
           2       0.64      0.60      0.62      1320

    accuracy                           0.76      4000
   macro avg       0.75      0.75      0.75      4000
weighted avg       0.75      0.76      0.75      4000


[[1185    1  174]
 [   5 1043  272]
 [ 291  235  794]]


In [7]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss'
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("="*30)
print("XGBoost")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

XGBoost
Accuracy : 0.94925

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1360
           1       0.97      0.96      0.96      1320
           2       0.92      0.93      0.92      1320

    accuracy                           0.95      4000
   macro avg       0.95      0.95      0.95      4000
weighted avg       0.95      0.95      0.95      4000


[[1305    0   55]
 [   0 1266   54]
 [  53   41 1226]]


In [8]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

y_pred = nb.predict(X_test)

print("="*30)
print("Naive Bayes")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Naive Bayes
Accuracy : 0.6785

              precision    recall  f1-score   support

           0       0.79      0.80      0.79      1360
           1       0.66      0.88      0.75      1320
           2       0.54      0.36      0.43      1320

    accuracy                           0.68      4000
   macro avg       0.66      0.68      0.66      4000
weighted avg       0.67      0.68      0.66      4000


[[1082   34  244]
 [   9 1157  154]
 [ 279  566  475]]


In [9]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

gb.fit(X_train, y_train)

y_pred = gb.predict(X_test)

print("="*30)
print("Gradient Boosting")
print("="*30)

print("Accuracy :", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Gradient Boosting
Accuracy : 0.91125

              precision    recall  f1-score   support

           0       0.94      0.92      0.93      1360
           1       0.94      0.93      0.94      1320
           2       0.86      0.88      0.87      1320

    accuracy                           0.91      4000
   macro avg       0.91      0.91      0.91      4000
weighted avg       0.91      0.91      0.91      4000


[[1252    0  108]
 [   0 1232   88]
 [  84   75 1161]]


In [10]:
from sklearn.metrics import accuracy_score
import pandas as pd

results = {
    "Logistic Regression": accuracy_score(y_test, lr.predict(X_test)),
    "Decision Tree": accuracy_score(y_test, dt.predict(X_test)),
    "Random Forest": accuracy_score(y_test, best_rf.predict(X_test)),
    "KNN": accuracy_score(y_test, knn.predict(X_test)),
    "SVM": accuracy_score(y_test, svm.predict(X_test)),
    "Naive Bayes": accuracy_score(y_test, nb.predict(X_test)),
    "Gradient Boosting": accuracy_score(y_test, gb.predict(X_test)),
    "XGBoost": accuracy_score(y_test, xgb.predict(X_test))
}

comparison = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
comparison = comparison.sort_values(by="Accuracy", ascending=False)

print(comparison)

                 Model  Accuracy
7              XGBoost   0.94925
6    Gradient Boosting   0.91125
2        Random Forest   0.90200
1        Decision Tree   0.82925
0  Logistic Regression   0.82850
4                  SVM   0.75550
5          Naive Bayes   0.67850
3                  KNN   0.67225
